# Compare Ad Effectiveness Using A/B Tests with Python Project

- [View Solution Notebook](./solution.html)
- [View Project Page](https://www.codecademy.com/projects/practice/compare-ad-effectiveness-using-a-b-tests-with-python)

## Task Group 1 - Import, Inspect, and Merge

### Task 1

Import the **pandas** and **numpy** libraries using their standard aliases. 

In [2]:
import pandas as pd
import numpy as np

### Task 2

Import the CSV file `users.csv` and assign it to the variable `users`. Preview the first five rows.

In [3]:
users = pd.read_csv("users.csv")
users

,user_id,timestamp,device_id,clicked
0,U790620,2022-01-01 00:23:33 Saturday,M003,False
1,U584867,2022-01-01 01:30:07 Saturday,M001,False
2,U128681,2022-01-01 01:30:14 Saturday,M002,True
3,U694898,2022-01-01 01:31:55 Saturday,M003,False
4,U456823,2022-01-01 03:18:25 Saturday,M001,False
...,...,...,...,...
15268,U440244,2022-12-31 22:35:32 Saturday,M001,False
15269,U816412,2022-12-31 22:50:53 Saturday,P005,False
15270,U598035,2022-12-31 23:39:59 Saturday,T001,False
15271,U746551,2022-12-31 23:46:15 Saturday,P006,False


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>What is the structure of this dataset? Toggle to check!</i></summary>

Each row in `users` corresponds to an individual session in which a user viewed an ad from a specific device and either clicked or did not click the ad.

Here's a quick summary of the columns:

- `user_id` - a unique identifier for each user
- `timestamp` - the date and time the user viewed the ad in the format: `<YYYY-MM-DD HH:MM:SS day_of_week>`
- `device_id` - a unique identifier for the device the user was using to view the ad
- `clicked` - `True` if the user clicked on the ad, otherwise `False`

</details>

### Task 3

Import the CSV file `advertisements.csv` and assign it to the variable `advertisements`. Preview the first five rows.

In [4]:
advertisements = pd.read_csv("advertisements.csv")
advertisements

,user_id,timestamp,ad_source,ad_version
0,U790620,2022-01-01 00:23:33 Saturday,Twitter,A
1,U584867,2022-01-01 01:30:07 Saturday,Google,B
2,U128681,2022-01-01 01:30:14 Saturday,TikTok,B
3,U694898,2022-01-01 01:31:55 Saturday,TikTok,A
4,U456823,2022-01-01 03:18:25 Saturday,Google,A
...,...,...,...,...
14739,U440244,2022-12-31 22:35:32 Saturday,Twitter,A
14740,U816412,2022-12-31 22:50:53 Saturday,Google,A
14741,U598035,2022-12-31 23:39:59 Saturday,Google,A
14742,U746551,2022-12-31 23:46:15 Saturday,Google,A


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>What is the structure of this dataset? Toggle to check!</i></summary>

Each row in `advertisements` contains a more detailed breakdown of the advertisement corresponding to each user session.

Here's a quick summary of the columns:

- `user_id` - a unique identifier for each user
- `timestamp` - the date and time the user viewed the ad in the format: `<YYYY-MM-DD HH:MM:SS day_of_week>`
- `ad_source` - the social media platform the user viewed the ad on
- `ad_version` - the version of the ad the user viewed where `A` is the original ad and `B` is the new ad

</details>

### Task 4

There are two `user_id` columns, one in each DataFrame. Let's do a quick check to see if they have the same number of unique users. If they don't, we'll know to be careful about this issue later on in our analysis.

Count the unique number of `user_id`s in `users` and separately in `advertisements`.

Print both counts.

In [5]:
users_unique_count = users['user_id'].nunique()
advertisement_unique_count = advertisements['user_id'].nunique()

print(users_unique_count)
print(advertisement_unique_count)

15122
14602


### Task 5

In order to effectively analyze the results of our A/B test, we'll need rows that contain user information from both DataFrames. Specifically, we'll need to know whether or not a user `clicked` on the ad in `users` and we'll need to know which `ad_version` the user viewed in `advertisements`. 

Merge `users` and `advertisements` on the `user_id` and `timestamp` columns. Make sure to only return rows that have data from both DataFrames.

Save the merged DataFrame to a variable named `users_ads` and preview the first five rows. 

In [6]:
users_ads = pd.merge(left = users, right = advertisements, on  = ['user_id', 'timestamp'], how = 'inner')
users_ads

,user_id,timestamp,device_id,clicked,ad_source,ad_version
0,U790620,2022-01-01 00:23:33 Saturday,M003,False,Twitter,A
1,U584867,2022-01-01 01:30:07 Saturday,M001,False,Google,B
2,U128681,2022-01-01 01:30:14 Saturday,M002,True,TikTok,B
3,U694898,2022-01-01 01:31:55 Saturday,M003,False,TikTok,A
4,U456823,2022-01-01 03:18:25 Saturday,M001,False,Google,A
...,...,...,...,...,...,...
14419,U440244,2022-12-31 22:35:32 Saturday,M001,False,Twitter,A
14420,U816412,2022-12-31 22:50:53 Saturday,P005,False,Google,A
14421,U598035,2022-12-31 23:39:59 Saturday,T001,False,Google,A
14422,U746551,2022-12-31 23:46:15 Saturday,P006,False,Google,A


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>Why do we use a merge that returns matching rows from both DataFrames? Toggle to check!</i></summary>

We saw in task 4 that the two DataFrames do not have the same number of unique `user_id`s, so there are some users that are missing advertisement data. Perhaps these are users who opted out of certain tracking cookies. In any case, we only want to work with users where we know which ad they saw.

We'll also need to merge on multiple columns in the case where users with the same `user_id` saw the ad multiple times. By also merging with `timestamp`, we now have data from each user's individual session.

</details>

## Task Group 2 - Use Aggregations to Calculate Click-Through Rate

### Task 6

To begin our exploration, let's count how many times each `ad_version` was viewed by users in `users_ads`. 

Group `users_ads` by `ad_version` and `count` the `user_id` column. Save the result to the variable `ad_view_count`, and rename the `user_id` column `num_views`.

Preview the final result.

In [7]:
ad_view_count = users_ads.groupby(['ad_version']).agg({'user_id':'count'})
ad_view_count.columns = ['num_views']
ad_view_count = ad_view_count.reset_index()
ad_view_count

,ad_version,num_views
0,A,7154
1,B,7270


### Task 7

There may be instances where individual users saw the ads more than once. Let's re-do the groupby from Task 6, but adding a column with the number of unique `user_id`s for each ad.

Modify `ad_view_count` to also return the count of unique users in `user_id`. Name this new column `nunique`.

Preview the modified `ad_view_count`. 

In [8]:
ad_view_count = users_ads.groupby(['ad_version']).agg({'user_id':['count','nunique']})
ad_view_count.columns = ['num_views','nunique']
ad_view_count = ad_view_count.reset_index()
print(ad_view_count)
users_ads = users_ads.drop_duplicates(subset='user_id', keep='first')

users_ads.head()

  ad_version  num_views  nunique
0          A       7154     7125
1          B       7270     7232


,user_id,timestamp,device_id,clicked,ad_source,ad_version
0,U790620,2022-01-01 00:23:33 Saturday,M003,False,Twitter,A
1,U584867,2022-01-01 01:30:07 Saturday,M001,False,Google,B
2,U128681,2022-01-01 01:30:14 Saturday,M002,True,TikTok,B
3,U694898,2022-01-01 01:31:55 Saturday,M003,False,TikTok,A
4,U456823,2022-01-01 03:18:25 Saturday,M001,False,Google,A


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>How should we address users who saw ads more than once? Toggle to check!</i></summary>

Users who either saw both ad versions or one version multiple times may introduce bias to our test. For example, we might encounter **familiarity bias** in which users become more likely to click an ad they have seen more frequently.
    
In this case, there aren't that many repeat viewers, so we won't address the issue directly. But feel free to remove these duplicates and re-run our analysis on your own! 

</details>

### Task 8

Group by `ad_version` and compute the **percentage** of users who clicked on each ad. This metric is known as the **click-through rate (CTR)** and is widely used as a performance metric. 

Save the resultin DataFrame to the variable `ad_ctr_pct`.

Rename the `clicked` column to `click_rate`.

Preview `ad_ctr_pct`.

In [9]:
ad_ctr_pct = users_ads.groupby('ad_version').agg({'clicked': 'mean'})
ad_ctr_pct = ad_ctr_pct.rename(columns={'clicked': 'click_rate'})
ad_ctr_pct = ad_ctr_pct.reset_index()

ad_ctr_pct['click_rate'] = ad_ctr_pct['click_rate'] * 100

ad_ctr_pct

,ad_version,click_rate
0,A,12.413599
1,B,19.366491


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>What did we learn about each ad version? Toggle to check!</i></summary>

Looking at results of our A/B test, we see that the new ad version `B` has a click-through rate of about 19.4% and the old ad version `A` has a click-through rate of about 12.4% (a +7% difference!). 

While a 7% difference may seem huge, we'll need to continue exploring the data to be more confident in our conclusions about the effectiveness of each advertisement. 

</details>

## Task Group 3: Compare Ad Performances by Social Media Platform

### Task 9

Let's break down click-through rate by social media platform.

Group `users_ads` by `ad_source` and `ad_version`, and compute the percent of `True` values in `clicked`.

Rename the column containing the percents `ctr` and view the full resulting DataFrame.

In [10]:
ad_source_ctr = users_ads.groupby(['ad_source', 'ad_version']).agg({'clicked': 'mean'})

ad_source_ctr = ad_source_ctr.rename(columns={'clicked': 'ctr'})

ad_source_ctr['ctr'] = ad_source_ctr['ctr'] * 100

ad_source_ctr = ad_source_ctr.reset_index()

ad_source_ctr

,ad_source,ad_version,ctr
0,Google,A,12.912347
1,Google,B,19.863326
2,Meta,A,12.922932
3,Meta,B,18.815490
4,TikTok,A,11.596958
5,TikTok,B,20.009528
6,Twitter,A,11.836735
7,Twitter,B,17.630465


### Task 10

It is a little hard to read the long-format table from Task 9.
 
Pivot `users_ads` to create a wide-format version of this table, and name it `ad_social`.

Print all rows of `ad_social`.

In [13]:
ad_social = pd.pivot_table(users_ads, index = 'ad_source' , columns= 'ad_version' , values = 'clicked' , aggfunc = 'mean')
ad_social

ad_version,A,B
ad_source,,
Google,0.129123,0.198633
Meta,0.129229,0.188155
TikTok,0.115970,0.200095
Twitter,0.118367,0.176305


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>What did we learn about the ad performances across various social media platforms? Toggle to check!</i></summary>

The click-through rates for ad version `B` consistently outperformed the click-through rates for ad version `A` across all social media platforms. This is more evidence to suggest that our new ad version `B` is more effective than the old ad version `A`.  

Ad version `B` performed the best on TikTok with over 20% of users clicking on the ad which might suggest that the newer advertisement was more effective towards the younger demographic that make up the majority of TikTok users. 
    
Based on this data alone, we can't conclude whether or not the difference in ad performances across social media platforms is statistically significant. To confirm our results, we'd need to use a hypothesis test, which you can learn in our statistics courses.

</details>

## Task Group 4 - Compare Ad Performances by Tech Device

### Task 11

Let's now investigate the click-through rates broken down by device (cell phone, tablet, etc.)

Import the CSV file `devices.csv` and assign it to the variable `devices`. Preview the full dataset.

How many different devices are there?

In [15]:
devices = pd.read_csv("devices.csv")
devices

,device_id,device_type,brand
0,M001,Mobile,Apple
1,M002,Mobile,Samsung
2,M003,Mobile,Google
3,M004,Mobile,Huawei
4,M005,Mobile,Xiaomi
5,M006,Mobile,vivo
6,P001,PC,Apple
7,P002,PC,Dell
8,P003,PC,HP
9,P004,PC,ASUS


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>What is the structure of this dataset? Toggle to check!</i></summary>

There are 22 different devices in the data. 

Here's a quick summary of the columns:

- `device_id` - a unique identifier of the device
- `device_type` - the type of device
- `brand` - the brand of the device

</details>

### Task 12

Let's now combine each user's ad information with their device information. 

Merge `users_ads` with `devices` using the `device_id` column. Make sure to return all of the rows in `users_ads`, along with matching device information if it exists.

Name the merged DataFrame `users_devices` and preview the first five rows.

In [19]:
users_devices = pd.merge(left = users_ads, right = devices, on = ['device_id'], how = 'left')
users_devices

,user_id,timestamp,device_id,clicked,ad_source,ad_version,device_type,brand
0,U790620,2022-01-01 00:23:33 Saturday,M003,False,Twitter,A,Mobile,Google
1,U584867,2022-01-01 01:30:07 Saturday,M001,False,Google,B,Mobile,Apple
2,U128681,2022-01-01 01:30:14 Saturday,M002,True,TikTok,B,Mobile,Samsung
3,U694898,2022-01-01 01:31:55 Saturday,M003,False,TikTok,A,Mobile,Google
4,U456823,2022-01-01 03:18:25 Saturday,M001,False,Google,A,Mobile,Apple
...,...,...,...,...,...,...,...,...
14282,U440244,2022-12-31 22:35:32 Saturday,M001,False,Twitter,A,Mobile,Apple
14283,U816412,2022-12-31 22:50:53 Saturday,P005,False,Google,A,PC,Lenovo
14284,U598035,2022-12-31 23:39:59 Saturday,T001,False,Google,A,Tablet,Apple
14285,U746551,2022-12-31 23:46:15 Saturday,P006,False,Google,A,PC,Other


### Task 13

Calculate the percentage of users who clicked on the advertisement based on their `device_type` and `ad_version` they viewed.

In [20]:
device_ad_ctr = users_devices.groupby(
    ['device_type', 'ad_version']
).agg({'clicked': 'mean'})

device_ad_ctr = device_ad_ctr.rename(
    columns={'clicked': 'ctr'}
)

device_ad_ctr['ctr'] = device_ad_ctr['ctr'] * 100

device_ad_ctr = device_ad_ctr.reset_index()

device_ad_ctr

,device_type,ad_version,ctr
0,Mobile,A,12.137733
1,Mobile,B,21.460801
2,PC,A,12.127894
3,PC,B,18.243243
4,Tablet,A,12.725958
5,Tablet,B,17.195208


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>What did we learn about the ad performances across tech devices? Toggle to check!</i></summary>

Once again, the click-through rates for ad version `B` consistently outperformed the click-through rates for ad version `A` across all tech devices. This is even more evidence to suggest that our new ad version `B` is more effective than the old ad version `A`.  

Interestingly, it looks like ad version `B` was more effective towards mobile users compared to PC and tablet users. Recall that TikTok was also particularly well-represented in click-throughs.

</details>

## Task Group 5 - Weekday and Weekend Performance by Device Type

### Task 14

Let's break our analysis down further by weekday versus weekend user behavior. 

There are various approaches we could use to explore this question, but let's practice using the **split-apply-combine (SAC)**  technique.

Recall, SAC involves a three-step process where we:
1. **Split** the dataset into one or more pieces
2. **Apply** a function or transformation to each piece
3. **Combine** the results from each piece

We've provided starter code below that creates a new column `day_of_week` extracted from the `timestamp` column indicating the day on which the user viewed the ad.

Count the number of users who viewed each ad, grouped by the day of the week and the ad version they saw.

In [22]:
# Create 'day_of_week' column
users_devices['day_of_week'] = users_devices['timestamp'].str.split(' ', expand=True)[2]

## YOUR SOLUTION HERE
day_ad_count = users_devices.groupby(
    ['day_of_week', 'ad_version']
).agg({'user_id': 'count'})

day_ad_count = day_ad_count.rename(columns={'user_id': 'num_views'})
day_ad_count = day_ad_count.reset_index()

day_ad_count

,day_of_week,ad_version,num_views
0,Friday,A,1053
1,Friday,B,1053
2,Monday,A,1008
3,Monday,B,1027
4,Saturday,A,1034
5,Saturday,B,984
6,Sunday,A,971
7,Sunday,B,1041
8,Thursday,A,1012
9,Thursday,B,987


### Task 15

**Split** `users_devices` into two DataFrames:
- `weekends` filtering for users who saw either ad on Saturday or Sunday
- `weekdays` filtering for users who saw either ad from Monday to Friday

Use the `|` operator, and see the hint for more of a refresher on Boolean masks!

Preview the first five rows in `weekdays`.

In [30]:
is_weekend = ((users_devices['day_of_week'] == 'Saturday')| (users_devices['day_of_week'] == 'Sunday'))
              
weekends = users_devices[is_weekend]
              
is_weekday = (
    (users_devices['day_of_week'] != 'Saturday') &
    (users_devices['day_of_week'] != 'Sunday')
)
weekdays = users_devices[is_weekday]
weekdays.head()

,user_id,timestamp,device_id,clicked,ad_source,ad_version,device_type,brand,day_of_week
72,U532019,2022-01-03 00:01:43 Monday,M004,False,Meta,B,Mobile,Huawei,Monday
73,U204646,2022-01-03 00:04:32 Monday,M001,False,Google,A,Mobile,Apple,Monday
74,U484678,2022-01-03 00:10:18 Monday,M003,False,Meta,A,Mobile,Google,Monday
75,U676966,2022-01-03 00:39:32 Monday,M002,True,Google,B,Mobile,Samsung,Monday
76,U145580,2022-01-03 00:45:24 Monday,M001,False,Google,A,Mobile,Apple,Monday


### Task 16

**Apply** aggregation functions.

Create a new DataFrame `weekday_ctr` that computes

- the total number of views on weekdays
- the percent of clicks on weekdays

grouped by `device_type` and `ad_version`.

Name the two columns `weekday_views` and `weekday_rate`, and then reset the index of the DataFrame `weekday_ctr`.

Print the full `weekday_ctr` DataFrame.

In [31]:
weekday_ctr = weekdays.groupby(
    ['device_type', 'ad_version']
).agg(
    weekday_views=('user_id', 'count'),
    weekday_rate=('clicked', 'mean')
)

weekday_ctr['weekday_rate'] = weekday_ctr['weekday_rate'] * 100

weekday_ctr = weekday_ctr.reset_index()

print(weekday_ctr.to_string(index=False))

device_type ad_version  weekday_views  weekday_rate
     Mobile          A           2493     12.635379
     Mobile          B           2568     21.183801
         PC          A           1298     12.788906
         PC          B           1273     18.853103
     Tablet          A           1001     14.185814
     Tablet          B           1007     16.484608


### Task 17

Create a new DataFrame `weekend_ctr` that computes

- the total number of clicks on weekends
- the percent of clicks on weekends

grouped by `device_type` and `ad_version`.

Name the two columns `weekend_views` and `weekend_rate`, and then reset the index of the DataFrame `weekend_ctr`.

Print the full `weekend_ctr` DataFrame.

In [32]:
weekend_ctr = weekends.groupby(
    ['device_type', 'ad_version']
).agg(
    weekend_views=('user_id', 'count'),
    weekend_rate=('clicked', 'mean')
)

weekend_ctr['weekend_rate'] = weekend_ctr['weekend_rate'] * 100

weekend_ctr = weekend_ctr.reset_index()

print(weekend_ctr.to_string(index=False))

device_type ad_version  weekend_views  weekend_rate
     Mobile          A            992     10.887097
     Mobile          B            978     22.188139
         PC          A            516     10.465116
         PC          B            503     16.699801
     Tablet          A            382      8.900524
     Tablet          B            412     18.932039


### Task 18

**Combine** `weekday_ctr` and `weekend_ctr` by merging on the `device_type` and `ad_version` columns into a DataFrame named `combined_ctr`. Make sure to return matching rows from both DataFrames.

Preview all of the click-through rates in `combined_ctr`.

In [34]:
combined_ctr = pd.merge(
    left=weekday_ctr,
    right=weekend_ctr,
    on=['device_type', 'ad_version'],
    how='inner'
)

print(combined_ctr.to_string())

  device_type ad_version  weekday_views  weekday_rate  weekend_views  weekend_rate
0      Mobile          A           2493     12.635379            992     10.887097
1      Mobile          B           2568     21.183801            978     22.188139
2          PC          A           1298     12.788906            516     10.465116
3          PC          B           1273     18.853103            503     16.699801
4      Tablet          A           1001     14.185814            382      8.900524
5      Tablet          B           1007     16.484608            412     18.932039


## Task 19

It looks like there are more weekday views overall, which we'd expect, but let's see if the average views per day changes from weekend to weekday.

Modify `weekday_views` and `weekend_views` in `combined_ctr` to display the average number of clicks per day.

In [35]:
combined_ctr['weekday_views'] = combined_ctr['weekday_views'] / 5

combined_ctr['weekend_views'] = combined_ctr['weekend_views'] / 2

print(combined_ctr.to_string(index=False))

device_type ad_version  weekday_views  weekday_rate  weekend_views  weekend_rate
     Mobile          A          498.6     12.635379          496.0     10.887097
     Mobile          B          513.6     21.183801          489.0     22.188139
         PC          A          259.6     12.788906          258.0     10.465116
         PC          B          254.6     18.853103          251.5     16.699801
     Tablet          A          200.2     14.185814          191.0      8.900524
     Tablet          B          201.4     16.484608          206.0     18.932039


<details>
    <summary style="display:list-item; font-size:16px; color:blue;"><i>What did we learn about the ad performances on weekdays and weekends? Toggle to check!</i></summary>

Again we see that the click-through rates for ad version `B` consistently outperformed the click-through rates for ad version `A` across all tech devices regardless of whether or not it was on a weekday or weekend.
    
In fact, it seems as though there's no real difference in views on weekends versus weekdays, so for the same number of views we are just getting an increased number of clicks on the weekends.

Interestingly, ad version `A` underperformed on weekends compared to weekdays across all device types but the performance of ad version `B` remained relatively similar. This is further evidence to suggest that version `B` is a more effective ad than version `A` because of its consistent performance throughout the week. 

Considering the overall analysis of the click-through rate performances of the two different ads, which version do you think we should continue marketing with? Version `A` or `B`??

</details>

So that's the end of the the project comparing ad performances using A/B test data. There's so much more to discover in this dataset like the click-through rates of individual days, months, or even the brand of each device. Feel free to add more cells below and continue exploring! Happy coding!

CTR by adversion

In [37]:
import matplotlib.pyplot as plt

ad_ctr = users_ads.groupby('ad_version')['clicked'].mean() * 100

plt.figure(figsize=(7, 5))
plt.bar(ad_ctr.index, ad_ctr.values, color=['skyblue', 'orange'])

plt.title('Click-Through Rate by Ad Version')
plt.xlabel('Ad Version')
plt.ylabel('CTR (%)')
plt.tight_layout()
plt.show() 

CTR by social media platform

In [38]:
ad_social_pct = ad_social * 100

ad_social_pct.plot(
    kind='bar',
    figsize=(9, 5),
    color=['skyblue', 'orange'],
    width=0.7
)

plt.title('CTR by Social Media Platform')
plt.xlabel('Social Media Platform')
plt.ylabel('CTR (%)')
plt.xticks(rotation=0)
plt.legend(title='Ad Version')
plt.tight_layout()
plt.show()

CTR by device type and ad version

In [39]:
pivot_device = device_ad_ctr.pivot(
    index='device_type',
    columns='ad_version',
    values='ctr'
)

pivot_device.plot(
    kind='bar',
    figsize=(9, 5),
    color=['skyblue', 'orange'],
    width=0.7
)

plt.title('CTR by Device Type and Ad Version')
plt.xlabel('Device Type')
plt.ylabel('CTR (%)')
plt.xticks(rotation=0)
plt.legend(title='Ad Version')
plt.tight_layout()
plt.show()

Average daily views: weekday vs. weekend

In [40]:
view_data = combined_ctr.groupby('ad_version')[
    ['weekday_views', 'weekend_views']
].mean()

view_data.plot(
    kind='bar',
    figsize=(9, 5),
    color=['steelblue', 'coral'],
    width=0.7
)

plt.title('Average Daily Views: Weekday vs Weekend')
plt.xlabel('Ad Version')
plt.ylabel('Average Views per Day')
plt.xticks(rotation=0)
plt.legend(['Weekday', 'Weekend'])
plt.tight_layout()
plt.show()